128099145 linhas	

764456 ids

2023/07/04 - 2024/07/03

In [0]:
%run /Workspace/Users/daniel.branco@cgi.com/Tese/00_Utils

In [0]:
df = spark.read.table("hive_metastore.default.working_event_log")
display(df.limit(5))
df.printSchema()

In [0]:
summary = df.agg(
    F.count("*").alias("n_rows"),
    F.countDistinct("TAG1").alias("n_ids")
)
display(summary)

In [0]:
display(
    df.groupBy("ID_EVDATE")
      .count()
      .orderBy(F.desc("count"))
)

In [0]:
n = df.count()

nulls = df.agg(*[
    F.sum(F.col(c).isNull().cast("int")).alias(f"null_{c}")
    for c in df.columns
])

nulls_pct = nulls.select(*[
    (F.col(c) / F.lit(n)).alias(c.replace("null_", "pct_null_"))
    for c in nulls.columns
])

display(nulls)
display(nulls_pct)

In [0]:
agg_exprs = [
    F.sum((F.trim(F.col(c)) == F.lit("0")).cast("int")).alias(c)
    for c in df.columns
]

zero_counts = df.agg(*agg_exprs)
display(zero_counts)

In [0]:
zero_pct = zero_counts.select(*[
    (F.col(c) / F.lit(n)).alias(c)
    for c in zero_counts.columns
])

display(zero_pct)

In [0]:
invalid = df_parsed.filter(
    F.col("EVDATE_clean").isNotNull() & F.col("EVDATE_ts").isNull()
)

display(invalid.select("EVDATE", "EVDATE_clean").limit(50))


In [0]:
df = df.withColumn("EVDATE", F.expr("try_to_timestamp(NULLIF(trim(EVDATE), ''))"))


In [0]:
summary = df.agg(
    F.count("*").alias("n_rows"),
    F.countDistinct("TAG1").alias("n_ids"),
    F.min("EVDATE").alias("min_date"),
    F.max("EVDATE").alias("max_date"),
)
display(summary)

In [0]:
target_catalog = "hive_metastore"     # change this
target_schema = "bronze"
target_table = "bronze_event_log"  # change this

full_name = f"{target_catalog}.{target_schema}.{target_table}"


In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(full_name)
)

display(spark.table(full_name).limit(20))
